# 宽币赠送：生成 SQL → 查库 → 处理 CSV 全流程

配套 `grant_coins.py`。一个批次目录形如 `<任务>/<日期>/`，里面每个 `<金额>.txt`
是「要发这么多宽币的 user_id 名单，一行一个」。本 notebook 把该批次目录里
**所有** `.txt` 一次处理完，标准化流程如下：

1. **生成 SQL**（第 1 节）：
   - 每个 `<金额>.txt` 生成 `<金额>.username.sql`：查 `bigauth__user` 拿 username（发币用名单）。
   - 整批合成一条 `balance.sql`：查 `kbb__userkbb` 拿当前 balance（对账、查重复充值）。
2. **手动**：把 SQL 拿到服务器数据库执行，导出 CSV 放回同目录：
   - username 结果 → `<金额>.username.csv`（含表头 `id,username`）
   - balance 结果 → `balance.csv`（整批合一，含表头 `user_id,balance`）
3. **处理 username CSV**（第 2 节）：由 `<金额>.username.csv` 逐行写出
   `<金额>_username.txt`（最终按用户名发币的名单）。
4. **对账 / 查重复充值**（第 3 节）：读整批 `balance.csv`，结合台账
   `coin_grant_ledger.json`，标出余额异常（疑似重复充值）的用户。

只改「配置」cell 的 `BATCH_DIR` 指到你要处理的批次目录，然后从上往下跑即可。

In [ ]:
from pathlib import Path
import csv, json, re

# ===== 配置：改这里 =====================================================
# 要处理的批次目录（grant_coins.py 产物）：<任务>/<日期>/
# 相对本 notebook 所在目录 files/grant_coins/。
BATCH_DIR = Path("初始礼包/2026-07-01")

# 表名 / 空间（查余额用）。
USER_TABLE = "bigauth__user"          # id, username
KBB_TABLE = "kbb__userkbb"            # user_id, balance
SPACE_ID = "00000000-0000-0000-0000-000000000000"

# 台账（grant_coins.py 写的），用于对账时算「本应发放的累计金额」。
LEDGER_FILE = Path("coin_grant_ledger.json")
# ======================================================================

assert BATCH_DIR.is_dir(), f"批次目录不存在: {BATCH_DIR.resolve()}"

def read_ids(txt: Path) -> list[str]:
    """读 <金额>.txt，返回去空、去重后的 user_id 列表（保序）。"""
    ids = [ln.strip() for ln in txt.read_text(encoding="utf-8").splitlines() if ln.strip()]
    return list(dict.fromkeys(ids))

# 批次里所有 <金额>.txt（排除中间产物 *_username.txt）。
TXT_FILES = sorted(
    p for p in BATCH_DIR.glob("*.txt")
    if re.fullmatch(r"\d+", p.stem)  # 文件名是纯数字金额
)
print(f"批次目录: {BATCH_DIR}")
for p in TXT_FILES:
    print(f"  {p.name}: {len(read_ids(p))} 个 user_id")
assert TXT_FILES, "该批次目录下没有 <金额>.txt"

## 1. 生成 SQL

- `<金额>.username.sql` —— 每个金额一条，查 username，用于最终按用户名发币。
- `balance.sql` —— 整批合成一条，查当前 `balance`，用于发前/发后对账、排查重复充值。

跑完把 SQL 拿到服务器执行，导出 CSV 回同目录（见下一节命名）。balance 结果统一存
`balance.csv`（整批一份）。

In [ ]:
def in_clause(ids: list[str]) -> str:
    return ", ".join(f"'{i}'" for i in ids)

# username SQL：每个 <金额>.txt 一条（发币按金额分名单）。
for txt in TXT_FILES:
    ids = read_ids(txt)
    username_sql = f"SELECT id, username FROM {USER_TABLE} WHERE id IN ({in_clause(ids)})"
    txt.with_suffix(".username.sql").write_text(username_sql, encoding="utf-8")
    print(f"{txt.name}: {len(ids)} 个 id -> {txt.stem}.username.sql")

# balance SQL：整批合一条（对账时一次查完，结果统一存 balance.csv）。
all_ids = list(dict.fromkeys(i for txt in TXT_FILES for i in read_ids(txt)))
balance_sql = (
    f"SELECT user_id, balance FROM {KBB_TABLE} "
    f"WHERE space_id = '{SPACE_ID}' AND user_id IN ({in_clause(all_ids)})"
)
(BATCH_DIR / "balance.sql").write_text(balance_sql, encoding="utf-8")
print(f"\n整批 {len(all_ids)} 个 id -> balance.sql")

print("\n下一步：把上面 SQL 拿到服务器执行，导出 CSV 放回同目录：")
print("  <金额>.username.sql -> <金额>.username.csv （表头 id,username）")
print("  balance.sql         -> balance.csv         （表头 user_id,balance，整批合一）")

## 2. 生成 username 名单（查库回来后再跑）

把服务器导出的每个 `<金额>.username.csv`（含表头 `id,username`）转成
`<金额>_username.txt`（最终按用户名发币的名单，一行一个）。

同时校验：CSV 里的用户数是否和 `<金额>.txt` 一致，缺失的 id 会打印出来
（可能是查库漏了或用户不存在）。

In [ ]:
for txt in TXT_FILES:
    csv_path = txt.with_suffix(".username.csv")
    if not csv_path.exists():
        print(f"[跳过] 还没有 {csv_path.name}，先查库导出")
        continue

    id_to_name: dict[str, str] = {}
    with csv_path.open(encoding="utf-8") as f:
        for row in csv.DictReader(f):
            uid = (row.get("id") or "").strip()
            name = (row.get("username") or "").strip()
            if uid and name:
                id_to_name[uid] = name

    ids = read_ids(txt)
    usernames = [id_to_name[i] for i in ids if i in id_to_name]
    missing = [i for i in ids if i not in id_to_name]

    out = txt.with_name(f"{txt.stem}_username.txt")
    out.write_text("\n".join(usernames) + "\n", encoding="utf-8")
    print(f"{txt.name}: {len(usernames)}/{len(ids)} 个 username -> {out.name}")
    if missing:
        tail = " ..." if len(missing) > 5 else ""
        print(f"  ⚠ CSV 里缺 {len(missing)} 个 id（查库漏了或用户不存在）: {missing[:5]}{tail}")

## 3. 余额对账 / 查重复充值（查库回来后再跑）

读整批 `balance.csv`（表头 `user_id,balance`，全部金额合一份），结合台账
`coin_grant_ledger.json`（各任务累计发放额）做两类检查：

- **疑似重复充值**：`balance > 台账累计应发`。用户只会花币（余额下降），
  若当前余额比台账记的「一共发过多少」还高，说明有账外/重复充值，重点排查。
- **本批是否已到账**：按 user_id 回查它所属金额批次，`balance >= 本批金额`，
  粗略确认这批币发下去了（不精确，用户可能已消费，仅作参考）。

输出一张按 user_id 的对账表，把疑似重复充值单独列出，并落一份 `reconcile.csv`。

In [ ]:
# 台账 {任务key: {user_id: 已发金额}} -> 每个 user_id 的累计应发额。
granted_total: dict[str, int] = {}
if LEDGER_FILE.exists():
    ledger = json.loads(LEDGER_FILE.read_text(encoding="utf-8"))
    for _task, users in ledger.items():
        for uid, amt in users.items():
            granted_total[uid] = granted_total.get(uid, 0) + int(amt)
    print(f"台账: {LEDGER_FILE.name}，覆盖 {len(granted_total)} 个 user_id")
else:
    print(f"⚠ 未找到台账 {LEDGER_FILE}，只做「本批是否到账」的粗查，不做重复充值判断")

# 每个 user_id 属于哪个金额批次（用于「本批是否到账」判断）。
uid_to_amount: dict[str, int] = {}
for txt in TXT_FILES:
    for uid in read_ids(txt):
        uid_to_amount.setdefault(uid, int(txt.stem))

# 整批余额统一存在 balance.csv（表头 user_id,balance）。
balance_csv = BATCH_DIR / "balance.csv"
rows = []
if not balance_csv.exists():
    print(f"[跳过] 还没有 {balance_csv.name}，先执行 balance.sql 导出")
else:
    with balance_csv.open(encoding="utf-8") as f:
        for row in csv.DictReader(f):
            uid = (row.get("user_id") or "").strip()
            if not uid:
                continue
            balance = float(row.get("balance") or 0)
            amount = uid_to_amount.get(uid)
            expected = granted_total.get(uid)
            rows.append({
                "user_id": uid,
                "batch_amount": amount,
                "balance": balance,
                "ledger_total": expected,
                "疑似重复充值": expected is not None and balance > expected,
                "本批已到账": amount is not None and balance >= amount,
            })

print(f"\n对账 {len(rows)} 条记录")
try:
    import pandas as pd
    recon = pd.DataFrame(rows)
    dup = recon[recon["疑似重复充值"]] if len(recon) else recon
    print(f"疑似重复充值: {len(dup)} 人")
    display(dup.sort_values("balance", ascending=False) if len(dup) else recon.head())
    out_csv = BATCH_DIR / "reconcile.csv"
    recon.to_csv(out_csv, index=False, encoding="utf-8")
    print(f"对账明细已写入 {out_csv}")
except ImportError:
    dup = [r for r in rows if r["疑似重复充值"]]
    print(f"疑似重复充值: {len(dup)} 人")
    for r in dup[:20]:
        print("  ", r)